# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform basic analysis on the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata object and print some high-level info
metadata_obj = dataset.metadata
print(f"Name: {getattr(metadata_obj, 'name', None)}")
print(f"Description: {getattr(metadata_obj, 'description', None)}")
print(f"Version: {getattr(metadata_obj, 'version', None)}\n")

## 2. Data Overview
Review available record sets and their field and column `@id`s. All entities are referenced by their `@id`. This helps us select structures to inspect or load.


In [ ]:
# List record sets and their fields by @id.
print('Available RecordSets:')
if hasattr(metadata_obj, 'record_sets'):
    record_sets = getattr(metadata_obj, 'record_sets')
    for rs in record_sets:
        print(f"  RecordSet @id: {getattr(rs, '@id', None)}  Name: {getattr(rs, 'name', None)}")
        if hasattr(rs, 'fields'):
            for f in getattr(rs, 'fields'):
                print(f"    Field @id: {getattr(f, '@id', None)}  Name: {getattr(f, 'name', None)} DataType: {getattr(f, 'data_type', None)}")
else:
    # If .record_sets doesn't exist, try .to_json()['recordSet']
    meta_json = metadata_obj.to_json() if hasattr(metadata_obj, 'to_json') else metadata_obj
    record_sets = meta_json.get('recordSet', [])
    if record_sets:
        for rs in record_sets:
            print(f"  RecordSet @id: {rs.get('@id', None)}  Name: {rs.get('name', None)}")
            for f in rs.get('field', []):
                print(f"    Field @id: {f.get('@id', None)}  Name: {f.get('name', None)} DataType: {f.get('dataType', None)}")
    else:
        print('No record sets found in metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame using its `@id`. For demonstration, we choose the primary record set that contains protein measurements. Adjust the variables based on the overview above.

In [ ]:
# Define the primary record set @id (replace with an actual @id found above)
RECORD_SET_ID = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # Example

# List all record sets
print('All available record set @id\'s:')
for rs in dataset.record_sets:
    print(rs['@id'])

# Load records for the record set
records = list(dataset.records(record_set=RECORD_SET_ID))

# Convert to DataFrame
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records. Columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data handling steps: filtering, normalization, grouping, etc. Below, fields are referenced by their `@id` column names as returned by the DataFrame.

In [ ]:
# Inspect DataFrame columns and select a numeric field by @id (edit if necessary)
NUMERIC_FIELD = 'MW_kDa'  # For example, molecular weight in kDa
GROUP_FIELD = 'SampleID'  # For example, you may group by sample

print('Available columns:', df.columns.tolist())

# Filter for high molecular weight proteins
threshold = 50
if NUMERIC_FIELD in df.columns:
    filtered_df = df[df[NUMERIC_FIELD].astype(float) > threshold]
    print(f"Filtered records with {NUMERIC_FIELD} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize molecular weight
    filtered_df[f"{NUMERIC_FIELD}_normalized"] = (filtered_df[NUMERIC_FIELD].astype(float) - filtered_df[NUMERIC_FIELD].astype(float).mean()) / filtered_df[NUMERIC_FIELD].astype(float).std()
    print(f"\nNormalized {NUMERIC_FIELD} for filtered records:")
    print(filtered_df[[NUMERIC_FIELD, f"{NUMERIC_FIELD}_normalized"]].head())
else:
    print(f"Field {NUMERIC_FIELD} not found in DataFrame.")

# Group by field
if GROUP_FIELD in df.columns and NUMERIC_FIELD in df.columns:
    grouped_df = filtered_df.groupby(GROUP_FIELD)[NUMERIC_FIELD].mean()
    print(f"\nMean {NUMERIC_FIELD} grouped by {GROUP_FIELD}:")
    print(grouped_df.head())
else:
    print(f"Cannot group: Columns {GROUP_FIELD} or {NUMERIC_FIELD} missing.")

## 5. Visualization
Visualize field distributions, such as the histogram of molecular weight, and the relationship between fields. Use matplotlib or seaborn as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of molecular weights
if NUMERIC_FIELD in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[NUMERIC_FIELD].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {NUMERIC_FIELD}")
    plt.xlabel("Molecular Weight (kDa)")
    plt.ylabel("Count")
    plt.show()

# Scatter plot: MW vs. another numeric field if available
OTHER_NUM_FIELD = None
for col in df.columns:
    if col != NUMERIC_FIELD and pd.api.types.is_numeric_dtype(df[col]):
        OTHER_NUM_FIELD = col
        break

if NUMERIC_FIELD in df.columns and OTHER_NUM_FIELD:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df[NUMERIC_FIELD].astype(float), y=df[OTHER_NUM_FIELD].astype(float))
    plt.xlabel(NUMERIC_FIELD)
    plt.ylabel(OTHER_NUM_FIELD)
    plt.title(f"{NUMERIC_FIELD} vs {OTHER_NUM_FIELD}")
    plt.show()
else:
    print('No second numeric field available for scatter plot.')

## 6. Conclusion
In this notebook, we used the Croissant schema and the `mlcroissant` library to explore the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells. 

- We examined the metadata and discovered the available record sets and their field `@id`s.
- We extracted the main record set as a pandas DataFrame.
- We demonstrated basic filtering, normalization, and grouping using the DataFrame, referencing fields by `@id`.
- We visualized numeric distributions.

This workflow allows for reproducible, schema-driven data analysis and can be adapted to other Croissant datasets and more advanced applications.
